# Friend A STEP 1 — honest 20-epoch LOEO validator

This is the single overnight entry point required by **Friend A — order of work** in `TEAM.md`.

It performs STEP 1 end to end:

1. trains the joint detector + linker **from scratch** on the fixed 126-video `6bba` side;
2. keeps all 12 `44b6` videos outside training and predicts them at checkpoints 5/10/15/20 and training `best`;
3. evaluates each graph set with the official metric and records score, edge Jaccard, node recall and `N_pred/N_est`;
4. evaluates the pilkwang primary and seed314159 secondary references with exactly the same prediction command;
5. selects the Friend A checkpoint by the official LOEO score, uploads it as `friend_a_exp02_loeo20/`, updates
   `experiments/RESULTS.md` and `CONTEXT.md`, commits, and pushes `detector-baseline-v2`.

The recipe is fixed: BCE, window 2, downsample 1,4,4, batch 4, cosine `1e-4 -> 1e-5`, seed 42,
`--max-detections 1024`, 20 full epochs, milestone checkpoints every 5 epochs. No pilkwang or team weight is loaded
for the Friend A model. Reference weights are used only after training for comparison.

## Safe stop and resume

- Press Jupyter's **Interrupt/Stop once**. The notebook terminates the child trainer and keeps
  `training_state_latest.pt`, written atomically after every completed epoch.
- Run all cells again later. Model weights, AdamW, cosine scheduler, AMP scaler, Torch/NumPy/CUDA RNG and the data-loader
  generator resume from the last completed epoch. An interrupted epoch is repeated.
- Completed checkpoint evaluations are also skipped on restart. The current prediction is repeated only if it did not
  finish and produce a valid 12-video metrics JSON.
- Do not delete `outputs/local_friend_a_exp02_loeo_scratch_20ep/active/` until STEP 1 is published.

Expected duration on the local RTX 5070 Ti is about 15 hours for training plus the checkpoint/reference predictions.
Do not submit anything to the leaderboard from this notebook.

Start Jupyter from `C:\Projects\CellTracker`, select the project Python 3.12 environment, and choose **Run All**.
The defaults perform the full run and publication. Useful overrides before Jupyter starts:

```powershell
$env:BIOHUB_DATA_DIR = "C:\Projects\CellTracker\train"
$env:BIOHUB_STEP1_AUTO_UPLOAD = "0"  # keep results local if Kaggle publication should be deferred
$env:BIOHUB_STEP1_AUTO_GIT = "0"     # keep report changes uncommitted if desired
jupyter lab
```


In [ ]:
from __future__ import annotations

import csv
import hashlib
import importlib.util
import json
import math
import os
import platform
import re
import shutil
import subprocess
import sys
import time
import zipfile
from datetime import datetime
from pathlib import Path


EXPERIMENT = "local_friend_a_exp02_loeo_scratch_20ep"
METHOD = "friend_a_exp02_loeo20_scratch_bce_w2"
WEIGHTS_DATASET = "vitaliineibauer/biohub-ft-weights"
WEIGHTS_DATASET_SUBFOLDER = "friend_a_exp02_loeo20"
EXPECTED_BRANCH = "detector-baseline-v2"

EPOCHS = 20
MILESTONE_EPOCHS = (5, 10, 15, 20)
BATCH_SIZE = 4
NUM_WORKERS = 0 if os.name == "nt" else 4
LEARNING_RATE = 1e-4
MIN_LEARNING_RATE = 1e-5
LR_SCHEDULE = "cosine"
SEED = 42
WINDOW_SIZE = 2
DET_LOSS = "bce"
DET_NEG_WEIGHT = 0.01
DET_LOSS_WEIGHT = 1.0
DOWNSAMPLE = "1,4,4"
POOL_KERNEL_UM = 5.0
MAX_DETECTIONS = 1024
CHECKPOINT_EVERY = 5
DET_THRESHOLD = 0.965
ILP_APPEARANCE = 0.0
ILP_DISAPPEARANCE = 2.0
ILP_DIVISION = 1.2

RUN_PIPELINE = os.environ.get("BIOHUB_STEP1_RUN_PIPELINE", "1") != "0"
AUTO_UPLOAD = os.environ.get("BIOHUB_STEP1_AUTO_UPLOAD", "1") != "0"
AUTO_GIT = os.environ.get("BIOHUB_STEP1_AUTO_GIT", "1") != "0"
RUN_ID = os.environ.get("BIOHUB_STEP1_RUN_ID", "active").strip() or "active"


def find_project_root(start: Path) -> Path:
    for candidate in [start.resolve(), *start.resolve().parents]:
        for repo_candidate in (candidate, candidate / ".github-push-worktree"):
            if (repo_candidate / "TEAM.md").exists() and (repo_candidate / "tools" / "repo" / "scripts").exists():
                return repo_candidate
    raise FileNotFoundError("Repository root not found. Start Jupyter inside C:\\Projects\\CellTracker.")


PROJECT_ROOT = find_project_root(Path.cwd())
WORKSPACE_ROOT = PROJECT_ROOT.parent if PROJECT_ROOT.name == ".github-push-worktree" else PROJECT_ROOT
GIT = ["git", "-c", f"safe.directory={PROJECT_ROOT.as_posix()}"]
explicit_data = os.environ.get("BIOHUB_DATA_DIR", "").strip()
data_candidates = [
    Path(explicit_data).expanduser() if explicit_data else None,
    WORKSPACE_ROOT / "train",
    PROJECT_ROOT / "biohub-cell-tracking-during-development" / "train",
    PROJECT_ROOT / "train",
]
DATA_DIR = next((path.resolve() for path in data_candidates if path is not None and path.exists()), None)
if DATA_DIR is None:
    checked = "\n".join(str(path) for path in data_candidates if path is not None)
    raise FileNotFoundError(f"Training data not found. Set BIOHUB_DATA_DIR. Checked:\n{checked}")

default_output_root = WORKSPACE_ROOT / "outputs"
OUTPUT_ROOT = Path(os.environ.get("BIOHUB_OUTPUT_DIR", default_output_root)).expanduser().resolve()
RUN_DIR = OUTPUT_ROOT / EXPERIMENT / RUN_ID
TRAIN_REPO = RUN_DIR / "train_repo"
TRAIN_OUTPUT = TRAIN_REPO / "weights" / METHOD / "split_0"
TRAIN_STATE = TRAIN_OUTPUT / "training_state_latest.pt"
ARTIFACT_DIR = RUN_DIR / "artifact"
PREDICTIONS_ROOT = RUN_DIR / "predictions"
EVALUATION_ROOT = RUN_DIR / "evaluation"
REFERENCE_ROOT = OUTPUT_ROOT / "_reference_weights"
PIPELINE_STATE_PATH = RUN_DIR / "pipeline_state.json"
RUN_CONFIG_PATH = RUN_DIR / "run_config.json"
EXPERIMENT_DIR = PROJECT_ROOT / "experiments" / EXPERIMENT
SPLIT_SOURCE = PROJECT_ROOT / "experiments" / "_lib" / "loeo_train6bba_val44b6.json"
TRAIN_PATCH_SOURCE = PROJECT_ROOT / "experiments" / "_lib" / "train_patches.py"


def sha256_file(path: Path) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()


def atomic_write_text(path: Path, text: str) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_suffix(path.suffix + ".tmp")
    temporary.write_text(text, encoding="utf-8")
    temporary.replace(path)


def load_pipeline_state() -> dict:
    if not PIPELINE_STATE_PATH.exists():
        return {"experiment": EXPERIMENT, "run_id": RUN_ID, "status": "new", "evaluations": {}}
    return json.loads(PIPELINE_STATE_PATH.read_text(encoding="utf-8"))


def save_pipeline_state(**updates) -> dict:
    state = load_pipeline_state()
    state.update(updates)
    state["updated_at"] = datetime.now().astimezone().isoformat()
    atomic_write_text(PIPELINE_STATE_PATH, json.dumps(state, ensure_ascii=False, indent=2))
    return state


def terminate_process(process: subprocess.Popen) -> None:
    if process.poll() is not None:
        return
    process.terminate()
    try:
        process.wait(timeout=30)
    except subprocess.TimeoutExpired:
        process.kill()
        process.wait(timeout=30)


def print_subprocess_line(line: str) -> None:
    encoding = getattr(sys.stdout, "encoding", None) or "utf-8"
    print(line.encode(encoding, errors="replace").decode(encoding, errors="replace"), end="", flush=True)


def run_streamed(command: list[str], *, cwd: Path, env: dict, log_path: Path, label: str) -> None:
    log_path.parent.mkdir(parents=True, exist_ok=True)
    print(f"\n[{label}] {subprocess.list2cmdline(command)}", flush=True)
    with log_path.open("a", encoding="utf-8") as log_handle:
        log_handle.write(f"\n===== {datetime.now().astimezone().isoformat()} | {label} =====\n")
        log_handle.flush()
        process = subprocess.Popen(
            command,
            cwd=cwd,
            env=env,
            stdout=subprocess.PIPE,
            stderr=subprocess.STDOUT,
            text=True,
            encoding="utf-8",
            errors="replace",
            bufsize=1,
        )
        try:
            assert process.stdout is not None
            for line in process.stdout:
                log_handle.write(line)
                log_handle.flush()
                print_subprocess_line(line)
            return_code = process.wait()
        except KeyboardInterrupt:
            terminate_process(process)
            save_pipeline_state(status=f"interrupted_during_{label}")
            print("\nStopped safely. Run all cells again to continue from the last completed unit.", flush=True)
            raise
        except BaseException:
            terminate_process(process)
            save_pipeline_state(status=f"notebook_error_during_{label}")
            raise
    if return_code != 0:
        save_pipeline_state(status=f"failed_during_{label}", return_code=return_code, failed_log=str(log_path))
        raise RuntimeError(f"{label} failed with exit code {return_code}; see {log_path}")


def run_checked(command: list[str], *, cwd: Path, env: dict | None = None) -> str:
    result = subprocess.run(
        command,
        cwd=cwd,
        env=env,
        text=True,
        encoding="utf-8",
        errors="replace",
        capture_output=True,
    )
    if result.returncode != 0:
        raise RuntimeError(
            f"Command failed ({result.returncode}): {subprocess.list2cmdline(command)}\n"
            + (result.stdout + result.stderr)[-6000:]
        )
    return result.stdout


print(f"project: {PROJECT_ROOT}")
print(f"data:    {DATA_DIR}")
print(f"run:     {RUN_DIR}")
print(
    f"STEP 1: scratch 6bba->44b6 | epochs={EPOCHS} batch={BATCH_SIZE} workers={NUM_WORKERS} "
    f"lr={LEARNING_RATE:g}->{MIN_LEARNING_RATE:g} ({LR_SCHEDULE}) checkpoints={MILESTONE_EPOCHS}"
)


In [ ]:
# Validate the honest split, create/reuse a persistent run directory, and prepare a resumable trainer.

required_modules = [
    "torch", "tracksdata", "geff", "polars", "rustworkx", "zarr", "tqdm",
    "scipy", "skimage", "dask", "networkx", "pandas", "pyscipopt", "ilpy", "blosc2",
]
missing_modules = [name for name in required_modules if importlib.util.find_spec(name) is None]
if missing_modules:
    raise ImportError("Missing STEP-1 packages: " + ", ".join(missing_modules))

import torch

if sys.version_info[:2] != (3, 12):
    raise RuntimeError(f"Python 3.12 is required; current interpreter: {sys.version.split()[0]}")
if not torch.cuda.is_available():
    raise RuntimeError("CUDA is unavailable. Select the Python 3.12 project kernel and restart Jupyter.")
probe = torch.zeros(1, device="cuda")
del probe
torch.cuda.empty_cache()
device_name = torch.cuda.get_device_name(0)
print(f"CUDA OK: torch={torch.__version__}, CUDA={torch.version.cuda}, device={device_name}")

split_manifest = json.loads(SPLIT_SOURCE.read_text(encoding="utf-8"))
train_ids = list(split_manifest["train"])
holdout_ids = list(split_manifest["holdout"])
if len(train_ids) != 126 or len(holdout_ids) != 12:
    raise RuntimeError(f"Unexpected manifest size: train={len(train_ids)}, holdout={len(holdout_ids)}")
if len(train_ids) != len(set(train_ids)) or len(holdout_ids) != len(set(holdout_ids)):
    raise RuntimeError("Duplicate IDs in the LOEO manifest")
if set(train_ids) & set(holdout_ids):
    raise RuntimeError("Leakage: train and holdout IDs overlap")
if any(not name.startswith("6bba_") for name in train_ids):
    raise RuntimeError("Leakage: every train video must be 6bba")
if any(not name.startswith("44b6_") for name in holdout_ids):
    raise RuntimeError("Leakage: every holdout video must be 44b6")
missing_paths = [
    str(DATA_DIR / f"{name}{suffix}")
    for name in train_ids + holdout_ids
    for suffix in (".zarr", ".geff")
    if not (DATA_DIR / f"{name}{suffix}").exists()
]
if missing_paths:
    raise FileNotFoundError(f"Missing {len(missing_paths)} data paths. First entries:\n" + "\n".join(missing_paths[:20]))
print(f"Leak-free split OK: {len(train_ids)} train 6bba | {len(holdout_ids)} holdout 44b6")

branch = run_checked([*GIT, "branch", "--show-current"], cwd=PROJECT_ROOT).strip()
if AUTO_GIT and branch != EXPECTED_BRANCH:
    raise RuntimeError(f"Auto-publish requires branch {EXPECTED_BRANCH!r}; current branch is {branch!r}")
dirty_before = run_checked([*GIT, "status", "--porcelain"], cwd=PROJECT_ROOT).strip()
if dirty_before:
    print("WARNING: working tree is not clean. Auto-commit will refuse unrelated paths at publication time:\n" + dirty_before)

RUN_DIR.mkdir(parents=True, exist_ok=True)
EVALUATION_ROOT.mkdir(parents=True, exist_ok=True)
PREDICTIONS_ROOT.mkdir(parents=True, exist_ok=True)
EXPERIMENT_DIR.mkdir(parents=True, exist_ok=True)

TRAIN_SPLIT = RUN_DIR / "loeo_train6bba_val44b6_for_trainer.json"
EVAL_SPLIT = RUN_DIR / "loeo_44b6_for_prediction.json"
atomic_write_text(TRAIN_SPLIT, json.dumps([{"split": 0, "train": train_ids, "test": holdout_ids}], indent=2))
atomic_write_text(EVAL_SPLIT, json.dumps([{"split": 0, "train": [], "test": holdout_ids}], indent=2))


def replace_once(source: str, old: str, new: str, label: str) -> str:
    count = source.count(old)
    if count != 1:
        raise RuntimeError(f"STEP-1 resume patch {label!r}: expected one occurrence, found {count}")
    return source.replace(old, new, 1)


def apply_step1_resume_patch(script_path: Path) -> None:
    source = script_path.read_text(encoding="utf-8")
    source = replace_once(
        source,
        "    init_weights: Path | None = None,\n) -> UNetNodeTransformer:\n",
        "    init_weights: Path | None = None,\n    resume_state: Path | None = None,\n) -> UNetNodeTransformer:\n",
        "train signature",
    )
    source = replace_once(
        source,
        '    parser.add_argument("--init-weights", type=str, default=None, help="Full model checkpoint to fine-tune from.")\n',
        '    parser.add_argument("--init-weights", type=str, default=None, help="Full model checkpoint to fine-tune from.")\n'
        '    parser.add_argument("--resume-state", type=str, default=None, help="Resume full model/optimizer/scheduler state.")\n',
        "resume CLI",
    )
    source = replace_once(
        source,
        "    args = parser.parse_args()\n",
        "    args = parser.parse_args()\n"
        "    if args.init_weights and args.resume_state:\n"
        "        raise ValueError(\"--init-weights and --resume-state are mutually exclusive\")\n",
        "resume CLI guard",
    )
    source = replace_once(
        source,
        "        train(\n            init_weights=Path(args.init_weights) if args.init_weights else None,\n            seed=args.seed,\n",
        "        train(\n            init_weights=Path(args.init_weights) if args.init_weights else None,\n"
        "            resume_state=Path(args.resume_state) if args.resume_state else None,\n            seed=args.seed,\n",
        "pass resume state",
    )
    source = replace_once(
        source,
        "    if init_weights is not None:\n",
        "    resume_payload = None\n"
        "    start_epoch = 0\n"
        "    resume_best_score = 0.0\n"
        "    if resume_state is not None:\n"
        "        resume_payload = torch.load(resume_state, map_location=\"cpu\", weights_only=False)\n"
        "        if resume_payload.get(\"format\") != \"friend_a_step1_state_v1\":\n"
        "            raise RuntimeError(f\"Unsupported resume-state format: {resume_payload.get('format')}\")\n"
        "        if resume_payload.get(\"model_config\") != model_config:\n"
        "            raise RuntimeError(\"Resume-state model config differs from the requested architecture\")\n"
        "        if int(resume_payload.get(\"target_epochs\", -1)) != n_epochs:\n"
        "            raise RuntimeError(\"Resume-state target epoch count differs from --epochs\")\n"
        "        start_epoch = int(resume_payload[\"completed_epochs\"])\n"
        "        if not 0 <= start_epoch <= n_epochs:\n"
        "            raise RuntimeError(f\"Invalid completed epoch count in resume state: {start_epoch}\")\n"
        "        resume_best_score = float(resume_payload[\"best_score\"])\n"
        "        _resume_missing, _resume_unexpected = model.load_state_dict(resume_payload[\"model_state\"], strict=True)\n"
        "        if _resume_missing or _resume_unexpected:\n"
        "            raise RuntimeError(\"Resume-state model keys do not match\")\n"
        "        print(f\"  loaded full resume state after epoch {start_epoch}/{n_epochs}\", flush=True)\n"
        "    if init_weights is not None:\n",
        "load resume model",
    )
    source = replace_once(
        source,
        "    print(\n        f\"Starting training for {n_epochs} epochs (batch_size={batch_size}, \"\n",
        "    if resume_payload is not None:\n"
        "        optimizer.load_state_dict(resume_payload[\"optimizer_state\"])\n"
        "        if scheduler is not None and resume_payload.get(\"scheduler_state\") is not None:\n"
        "            scheduler.load_state_dict(resume_payload[\"scheduler_state\"])\n"
        "        if resume_payload.get(\"scaler_state\") is not None:\n"
        "            SCALER.load_state_dict(resume_payload[\"scaler_state\"])\n"
        "        if g is not None and resume_payload.get(\"dataloader_generator_state\") is not None:\n"
        "            g.set_state(resume_payload[\"dataloader_generator_state\"])\n"
        "        if resume_payload.get(\"torch_rng_state\") is not None:\n"
        "            torch.set_rng_state(resume_payload[\"torch_rng_state\"])\n"
        "        if device.type == \"cuda\" and resume_payload.get(\"cuda_rng_state_all\") is not None:\n"
        "            torch.cuda.set_rng_state_all(resume_payload[\"cuda_rng_state_all\"])\n"
        "        if resume_payload.get(\"numpy_rng_state\") is not None:\n"
        "            np.random.set_state(resume_payload[\"numpy_rng_state\"])\n"
        "        del resume_payload\n"
        "    print(\n        f\"Starting training for {n_epochs} epochs (batch_size={batch_size}, \"\n",
        "restore optimiser and RNG",
    )
    source = replace_once(
        source,
        "    best_score = 0.0\n",
        "    best_score = resume_best_score\n",
        "resume best score",
    )
    source = replace_once(
        source,
        '    pbar = tqdm(range(n_epochs), desc="Training", disable=False)\n',
        '    pbar = tqdm(range(start_epoch, n_epochs), total=n_epochs, initial=start_epoch, desc="Training", disable=False)\n',
        "resume epoch range",
    )
    source = replace_once(
        source,
        "            rng = np.random.default_rng()\n",
        "            rng = np.random.default_rng(int(np.random.randint(0, 2**32, dtype=np.uint64)))\n",
        "reproducible augmentation RNG",
    )
    source = replace_once(
        source,
        '            f"  Epoch {epoch:3d}/{n_epochs} | lr={current_lr:.2e} | edge={edge_loss:.4f} | det={det_loss:.4f} | "\n',
        '            f"  Epoch {epoch + 1:3d}/{n_epochs} | lr={current_lr:.2e} | edge={edge_loss:.4f} | det={det_loss:.4f} | "\n',
        "one-based epoch display",
    )
    source = replace_once(
        source,
        "        if scheduler is not None:\n            scheduler.step()\n        marker = \"*\" if is_best else \" \"\n",
        "        if scheduler is not None:\n"
        "            scheduler.step()\n"
        "        _resume_model_state = {k.replace(\"unet.module.\", \"unet.\", 1): v for k, v in model.state_dict().items()}\n"
        "        _resume_payload = {\n"
        "            \"format\": \"friend_a_step1_state_v1\",\n"
        "            \"completed_epochs\": epoch + 1,\n"
        "            \"target_epochs\": n_epochs,\n"
        "            \"best_score\": best_score,\n"
        "            \"model_config\": model_config,\n"
        "            \"model_state\": _resume_model_state,\n"
        "            \"optimizer_state\": optimizer.state_dict(),\n"
        "            \"scheduler_state\": scheduler.state_dict() if scheduler is not None else None,\n"
        "            \"scaler_state\": SCALER.state_dict(),\n"
        "            \"torch_rng_state\": torch.get_rng_state(),\n"
        "            \"cuda_rng_state_all\": torch.cuda.get_rng_state_all() if device.type == \"cuda\" else None,\n"
        "            \"numpy_rng_state\": np.random.get_state(),\n"
        "            \"dataloader_generator_state\": g.get_state() if g is not None else None,\n"
        "        }\n"
        "        _resume_tmp = output_dir / \"training_state_latest.tmp\"\n"
        "        torch.save(_resume_payload, _resume_tmp)\n"
        "        _resume_tmp.replace(output_dir / \"training_state_latest.pt\")\n"
        "        print(f\"  Resume state saved after epoch {epoch + 1}/{n_epochs}\", flush=True)\n"
        "        marker = \"*\" if is_best else \" \"\n",
        "atomic full-state checkpoint",
    )
    script_path.write_text(source, encoding="utf-8")
    print(f"Applied STEP-1 full-state resume patch to {script_path}")


patch_marker = TRAIN_REPO / ".friend_a_step1_patch_complete"
if TRAIN_REPO.exists() and not patch_marker.exists():
    if not TRAIN_REPO.resolve().is_relative_to(RUN_DIR.resolve()):
        raise RuntimeError(f"Refusing to replace unexpected training repository: {TRAIN_REPO}")
    shutil.rmtree(TRAIN_REPO)

if not patch_marker.exists():
    TRAIN_REPO.mkdir(parents=True)
    for subdir in ("scripts", "src"):
        shutil.copytree(PROJECT_ROOT / "tools" / "repo" / subdir, TRAIN_REPO / subdir)
    patch_spec = importlib.util.spec_from_file_location("friend_a_train_patches", TRAIN_PATCH_SOURCE)
    if patch_spec is None or patch_spec.loader is None:
        raise RuntimeError(f"Cannot import {TRAIN_PATCH_SOURCE}")
    patch_module = importlib.util.module_from_spec(patch_spec)
    patch_spec.loader.exec_module(patch_module)
    train_script = TRAIN_REPO / "scripts" / "train_unet_transformer.py"
    patch_module.apply_train_patches(train_script)
    trainer_source = train_script.read_text(encoding="utf-8")
    legacy_import = "from biohub_tracking.models import SimpleNodeTransformer, TemporalUNet3D"
    if legacy_import in trainer_source:
        trainer_source = trainer_source.replace(
            legacy_import,
            "from biohub_tracking.models.simple_node_transformer import SimpleNodeTransformer\n"
            "from biohub_tracking.models.temporal_unet import TemporalUNet3D",
            1,
        )
        train_script.write_text(trainer_source, encoding="utf-8")
    apply_step1_resume_patch(train_script)
    atomic_write_text(patch_marker, "friend_a_step1_state_v1\n")

TRAIN_SCRIPT = TRAIN_REPO / "scripts" / "train_unet_transformer.py"
PREDICT_SCRIPT = TRAIN_REPO / "scripts" / "predict_unet_transformer.py"
compile(TRAIN_SCRIPT.read_text(encoding="utf-8"), str(TRAIN_SCRIPT), "exec")
compile(PREDICT_SCRIPT.read_text(encoding="utf-8"), str(PREDICT_SCRIPT), "exec")
help_env = {
    **os.environ,
    "PYTHONPATH": str(TRAIN_REPO / "src"),
    "BIOHUB_DATA_DIR": str(DATA_DIR),
    "PYTHONUTF8": "1",
    "PYTHONIOENCODING": "utf-8",
}
help_text = run_checked([sys.executable, str(TRAIN_SCRIPT), "--help"], cwd=TRAIN_REPO, env=help_env)
for flag in (
    "--resume-state", "--amp", "--no-grad-ckpt", "--seed", "--det-loss", "--max-detections",
    "--lr-schedule", "--min-lr", "--checkpoint-every",
):
    if flag not in help_text:
        raise RuntimeError(f"Prepared trainer is missing required flag: {flag}")

run_config = {
    "experiment": EXPERIMENT,
    "step": 1,
    "run_id": RUN_ID,
    "project_root": str(PROJECT_ROOT),
    "data_dir": str(DATA_DIR),
    "branch": branch,
    "python": sys.version,
    "platform": platform.platform(),
    "torch": torch.__version__,
    "cuda_runtime": torch.version.cuda,
    "cuda_device": device_name,
    "split": str(SPLIT_SOURCE.relative_to(PROJECT_ROOT)),
    "split_sha256": sha256_file(SPLIT_SOURCE),
    "train_videos": train_ids,
    "validation_videos": holdout_ids,
    "initialization": "scratch",
    "epochs": EPOCHS,
    "milestone_epochs": list(MILESTONE_EPOCHS),
    "batch_size": BATCH_SIZE,
    "num_workers": NUM_WORKERS,
    "learning_rate": LEARNING_RATE,
    "learning_rate_schedule": LR_SCHEDULE,
    "minimum_learning_rate": MIN_LEARNING_RATE,
    "seed": SEED,
    "window_size": WINDOW_SIZE,
    "det_loss": DET_LOSS,
    "det_neg_weight": DET_NEG_WEIGHT,
    "det_loss_weight": DET_LOSS_WEIGHT,
    "downsample": [int(value) for value in DOWNSAMPLE.split(",")],
    "pool_kernel_um": POOL_KERNEL_UM,
    "max_detections": MAX_DETECTIONS,
    "checkpoint_every": CHECKPOINT_EVERY,
    "amp": True,
    "gradient_checkpointing": False,
    "prediction": {
        "det_threshold": DET_THRESHOLD,
        "use_ilp": True,
        "ilp_appearance_weight": ILP_APPEARANCE,
        "ilp_disappearance_weight": ILP_DISAPPEARANCE,
        "ilp_division_weight": ILP_DIVISION,
        "post_processing": None,
    },
    "references": [
        "pilkwang/biohub-tracking-support-pack-50ep-v1",
        "pilkwang/biohub-temporal-unet3d-seed314159-v1",
    ],
    "weights_dataset_subfolder": WEIGHTS_DATASET_SUBFOLDER,
}
config_fingerprint = hashlib.sha256(json.dumps(run_config, sort_keys=True).encode("utf-8")).hexdigest()
run_config["config_fingerprint"] = config_fingerprint
if RUN_CONFIG_PATH.exists():
    previous_config = json.loads(RUN_CONFIG_PATH.read_text(encoding="utf-8"))
    if previous_config.get("config_fingerprint") != config_fingerprint:
        raise RuntimeError(
            f"Existing run {RUN_DIR} uses a different configuration. Set BIOHUB_STEP1_RUN_ID to a new value."
        )
else:
    atomic_write_text(RUN_CONFIG_PATH, json.dumps(run_config, ensure_ascii=False, indent=2))
save_pipeline_state(status="prepared", config_fingerprint=config_fingerprint)

REFERENCE_SPECS = {
    "reference_pilkwang_primary": "pilkwang/biohub-tracking-support-pack-50ep-v1",
    "reference_pilkwang_secondary_seed314159": "pilkwang/biohub-temporal-unet3d-seed314159-v1",
}
REFERENCE_REMOTE_DIR = "weights/unet_transformer/split_0"


def ensure_reference_model(label: str, dataset: str) -> Path:
    target_dir = REFERENCE_ROOT / label
    target_weights = target_dir / "edge_predictor_best.pth"
    target_config = target_dir / "config.json"
    if target_weights.exists() and target_config.exists():
        return target_weights
    target_dir.mkdir(parents=True, exist_ok=True)
    download_dir = target_dir / "download"
    download_dir.mkdir(parents=True, exist_ok=True)
    for filename in ("edge_predictor_best.pth", "config.json"):
        remote_file = f"{REFERENCE_REMOTE_DIR}/{filename}"
        output = run_checked(
            [
                sys.executable, "-m", "kaggle", "datasets", "download", dataset,
                "-f", remote_file, "-p", str(download_dir), "--unzip", "-o",
            ],
            cwd=PROJECT_ROOT,
        )
        if output.strip():
            print(output.strip())
        candidates = [path for path in download_dir.rglob(filename) if path.is_file()]
        if not candidates:
            raise FileNotFoundError(f"Kaggle download did not produce {remote_file} for {dataset}")
        shutil.copy2(candidates[0], target_dir / filename)
    state = torch.load(target_weights, map_location="cpu", weights_only=True)
    if not isinstance(state, dict) or not state:
        raise RuntimeError(f"Invalid reference checkpoint: {target_weights}")
    del state
    print(f"Reference ready: {label} | {sha256_file(target_weights)}")
    return target_weights

print("STEP-1 preflight passed. The notebook is safe to resume in the same RUN_DIR.")


In [ ]:
# Train from scratch, or resume the full state from the last completed epoch.


def read_training_progress() -> tuple[int, dict | None]:
    if not TRAIN_STATE.exists():
        return 0, None
    payload = torch.load(TRAIN_STATE, map_location="cpu", weights_only=False)
    if payload.get("format") != "friend_a_step1_state_v1":
        raise RuntimeError(f"Unexpected training-state format in {TRAIN_STATE}")
    if int(payload.get("target_epochs", -1)) != EPOCHS:
        raise RuntimeError(f"Training-state target does not equal {EPOCHS} epochs")
    if payload.get("model_config") != {
        "unet_out_channels": 32,
        "unet_layers": [32, 64, 128],
        "downsample": [1, 4, 4],
        "window_size": WINDOW_SIZE,
        "pool_kernel_um": POOL_KERNEL_UM,
    }:
        raise RuntimeError("Training-state architecture differs from the fixed STEP-1 recipe")
    completed = int(payload["completed_epochs"])
    if not 0 <= completed <= EPOCHS:
        raise RuntimeError(f"Invalid completed epoch count: {completed}")
    return completed, payload


TRAIN_CMD_BASE = [
    sys.executable,
    str(TRAIN_SCRIPT),
    "--method", METHOD,
    "--data-dir", str(DATA_DIR),
    "--splits", str(TRAIN_SPLIT),
    "--split", "0",
    "--epochs", str(EPOCHS),
    "--batch-size", str(BATCH_SIZE),
    "--num-workers", str(NUM_WORKERS),
    "--lr", str(LEARNING_RATE),
    "--lr-schedule", LR_SCHEDULE,
    "--min-lr", str(MIN_LEARNING_RATE),
    "--det-loss-weight", str(DET_LOSS_WEIGHT),
    "--det-neg-weight", str(DET_NEG_WEIGHT),
    "--window-size", str(WINDOW_SIZE),
    "--downsample", DOWNSAMPLE,
    "--pool-kernel-um", str(POOL_KERNEL_UM),
    "--max-detections", str(MAX_DETECTIONS),
    "--checkpoint-every", str(CHECKPOINT_EVERY),
    "--det-loss", DET_LOSS,
    "--seed", str(SEED),
    "--amp",
    "--no-grad-ckpt",
    "--single-gpu",
]
if "--init-weights" in TRAIN_CMD_BASE or "--unet-weights" in TRAIN_CMD_BASE:
    raise RuntimeError("Leakage guard: STEP 1 must start from scratch")

if RUN_PIPELINE:
    completed_epochs, training_payload = read_training_progress()
    if training_payload is not None:
        del training_payload
    if completed_epochs < EPOCHS:
        train_command = [*TRAIN_CMD_BASE]
        if completed_epochs:
            train_command.extend(["--resume-state", str(TRAIN_STATE)])
            print(f"Resuming the exact full state after epoch {completed_epochs}/{EPOCHS}")
        else:
            print("Starting the honest Friend A model from scratch")
        atomic_write_text(RUN_DIR / "train_command.txt", subprocess.list2cmdline(train_command) + "\n")
        trainer_env = {
            **os.environ,
            "BIOHUB_DATA_DIR": str(DATA_DIR),
            "PYTHONPATH": str(TRAIN_REPO / "src"),
            "PYTHONUNBUFFERED": "1",
            "PYTHONUTF8": "1",
            "PYTHONIOENCODING": "utf-8",
            "PYTORCH_ALLOC_CONF": "expandable_segments:True",
        }
        save_pipeline_state(status="training", completed_epochs_before_launch=completed_epochs)
        run_streamed(
            train_command,
            cwd=TRAIN_REPO,
            env=trainer_env,
            log_path=RUN_DIR / "train.log",
            label="training",
        )
    completed_epochs, training_payload = read_training_progress()
    if completed_epochs != EPOCHS:
        raise RuntimeError(f"Trainer returned without reaching {EPOCHS} epochs: completed={completed_epochs}")
    del training_payload

    required_model_files = [
        TRAIN_OUTPUT / "edge_predictor_best.pth",
        TRAIN_OUTPUT / "config.json",
        *[TRAIN_OUTPUT / f"edge_predictor_epoch_{epoch:03d}.pth" for epoch in MILESTONE_EPOCHS],
    ]
    missing_model_files = [str(path) for path in required_model_files if not path.is_file()]
    if missing_model_files:
        raise FileNotFoundError("Training finished but checkpoint files are missing:\n" + "\n".join(missing_model_files))

    epoch_pattern = re.compile(
        r"Epoch\s+(?P<epoch>\d+)/(?P<epochs>\d+).*?lr=(?P<lr>[0-9.eE+-]+).*?"
        r"edge=(?P<edge>[0-9.eE+-]+).*?det=(?P<det>[0-9.eE+-]+).*?"
        r"test_loss=(?P<test_loss>[0-9.eE+-]+).*?acc=(?P<acc>[0-9.eE+-]+).*?"
        r"recall=(?P<recall>[0-9.eE+-]+).*?best=(?P<best>[0-9.eE+-]+)(?:\s+(?P<marker>\*))?\s+\|\s*"
        r"train=(?P<train_seconds>[0-9.eE+-]+)s\s+test=(?P<validation_seconds>[0-9.eE+-]+)s"
    )
    epoch_by_number = {}
    for match in epoch_pattern.finditer((RUN_DIR / "train.log").read_text(encoding="utf-8", errors="replace")):
        epoch_number = int(match.group("epoch"))
        epoch_by_number[epoch_number] = {
            "epoch": epoch_number,
            **{
                key: float(match.group(key))
                for key in (
                    "lr", "edge", "det", "test_loss", "acc", "recall", "best",
                    "train_seconds", "validation_seconds",
                )
            },
            "checkpoint_saved_as_internal_best": match.group("marker") == "*",
        }
    training_history = [epoch_by_number[number] for number in sorted(epoch_by_number)]
    if [row["epoch"] for row in training_history] != list(range(1, EPOCHS + 1)):
        raise RuntimeError("Could not reconstruct a complete 1..20 training history from train.log")
    atomic_write_text(RUN_DIR / "training_history.json", json.dumps(training_history, indent=2))
    history_fields = list(training_history[0])
    with (RUN_DIR / "training_history.csv").open("w", newline="", encoding="utf-8") as handle:
        writer = csv.DictWriter(handle, fieldnames=history_fields)
        writer.writeheader()
        writer.writerows(training_history)
    internal_best_rows = [row for row in training_history if row["checkpoint_saved_as_internal_best"]]
    internal_best_epoch = internal_best_rows[-1]["epoch"] if internal_best_rows else None
    save_pipeline_state(
        status="training_complete",
        completed_epochs=EPOCHS,
        internal_best_epoch=internal_best_epoch,
        internal_best_score=max(row["best"] for row in training_history),
    )
    print(f"Training complete: {EPOCHS}/{EPOCHS} epochs; internal best epoch={internal_best_epoch}")
else:
    print("BIOHUB_STEP1_RUN_PIPELINE=0: training intentionally skipped after preflight.")


In [ ]:
# Predict and officially score milestones, training-best, and the two pilkwang references.


def load_valid_metrics(path: Path) -> dict | None:
    if not path.is_file():
        return None
    payload = json.loads(path.read_text(encoding="utf-8"))
    rows = payload.get("rows", [])
    summary = payload.get("summary", {})
    if int(summary.get("n", -1)) != len(holdout_ids):
        return None
    if {row.get("dataset") for row in rows} != set(holdout_ids):
        return None
    return payload


def evaluation_is_complete(payload: dict | None) -> bool:
    prediction_zip = payload.get("prediction_zip") if payload is not None else None
    return bool(
        payload is not None
        and "checkpoint" in payload
        and "aggregate_node_counts" in payload
        and prediction_zip
        and Path(prediction_zip).is_file()
    )


def n_pred_over_n_est(rows: list[dict]) -> tuple[int, float, float]:
    from geff import GeffMetadata

    predicted = 0
    estimated = 0.0
    for row in rows:
        n_pred = int(row["num_pred_nodes"])
        metadata = GeffMetadata.read(DATA_DIR / f"{row['dataset']}.geff")
        n_est = float((metadata.extra or {}).get("estimated_number_of_nodes", float("nan")))
        if not math.isfinite(n_est) or n_est <= 0:
            raise RuntimeError(f"Missing estimated_number_of_nodes for {row['dataset']}")
        expected_ratio = (n_pred - n_est) / n_est
        reported_ratio = float(row["total_node_ratio"])
        if not math.isclose(expected_ratio, reported_ratio, rel_tol=1e-9, abs_tol=1e-9):
            raise RuntimeError(f"Official scorer node ratio mismatch for {row['dataset']}")
        predicted += n_pred
        estimated += n_est
    return predicted, estimated, predicted / estimated


def zip_and_remove_predictions(prediction_dir: Path, label: str) -> Path:
    if not prediction_dir.resolve().is_relative_to(PREDICTIONS_ROOT.resolve()):
        raise RuntimeError(f"Refusing to archive unexpected prediction path: {prediction_dir}")
    zip_dir = RUN_DIR / "prediction_zips"
    zip_dir.mkdir(parents=True, exist_ok=True)
    zip_path = zip_dir / f"{label}.zip"
    temporary_base = zip_dir / f".{label}.building"
    temporary_zip = Path(shutil.make_archive(str(temporary_base), "zip", root_dir=prediction_dir))
    temporary_zip.replace(zip_path)
    shutil.rmtree(prediction_dir)
    return zip_path


def evaluate_weights(label: str, weights_path: Path, *, candidate: bool, epoch: int | None) -> dict:
    evaluation_dir = EVALUATION_ROOT / label
    evaluation_dir.mkdir(parents=True, exist_ok=True)
    metrics_path = evaluation_dir / "official_metrics.json"
    existing = load_valid_metrics(metrics_path)
    if evaluation_is_complete(existing):
        print(f"Skipping completed evaluation: {label} | score={existing['summary']['score']:.4f}")
        return existing

    if not weights_path.is_file() or not weights_path.with_name("config.json").is_file():
        raise FileNotFoundError(f"Checkpoint/config pair missing for {label}: {weights_path}")
    method = f"step1_{label}"
    prediction_dir = PREDICTIONS_ROOT / "friend_a_step1" / method / "split_0"
    predictor_env = {
        **os.environ,
        "USER": "friend_a_step1",
        "BIOHUB_DATA_DIR": str(DATA_DIR),
        "BIOHUB_PREDICTIONS_DIR": str(PREDICTIONS_ROOT),
        "PYTHONPATH": str(TRAIN_REPO / "src"),
        "PYTHONUNBUFFERED": "1",
        "PYTHONUTF8": "1",
        "PYTHONIOENCODING": "utf-8",
        "PYTORCH_ALLOC_CONF": "expandable_segments:True",
    }
    predict_command = [
        sys.executable,
        str(PREDICT_SCRIPT),
        "--method", method,
        "--data-dir", str(DATA_DIR),
        "--splits", str(EVAL_SPLIT),
        "--split", "0",
        "--weights", str(weights_path),
        "--unet-batch-size", "4",
        "--det-threshold", str(DET_THRESHOLD),
        "--use-ilp",
        "--ilp-appearance-weight", str(ILP_APPEARANCE),
        "--ilp-disappearance-weight", str(ILP_DISAPPEARANCE),
        "--ilp-division-weight", str(ILP_DIVISION),
    ]
    started = time.monotonic()
    save_pipeline_state(status=f"evaluating_{label}", active_evaluation=label)
    run_streamed(
        predict_command,
        cwd=TRAIN_REPO,
        env=predictor_env,
        log_path=evaluation_dir / "predict.log",
        label=f"predict_{label}",
    )
    geffs = sorted(prediction_dir.glob("*.geff"))
    if len(geffs) != len(holdout_ids) or {path.stem for path in geffs} != set(holdout_ids):
        raise RuntimeError(f"Prediction coverage for {label} is not exactly the 12 LOEO videos")

    score_env = {
        **os.environ,
        "BIOHUB_DATA_DIR": str(DATA_DIR),
        "PYTHONUNBUFFERED": "1",
        "PYTHONUTF8": "1",
        "PYTHONIOENCODING": "utf-8",
    }
    score_command = [
        sys.executable,
        str(PROJECT_ROOT / "tools" / "score_geffs.py"),
        str(prediction_dir),
        "--out", str(metrics_path),
    ]
    run_streamed(
        score_command,
        cwd=PROJECT_ROOT,
        env=score_env,
        log_path=evaluation_dir / "score.log",
        label=f"score_{label}",
    )
    metrics = load_valid_metrics(metrics_path)
    if metrics is None:
        raise RuntimeError(f"Official scorer did not create a complete metrics file for {label}")
    n_pred, n_est, node_ratio = n_pred_over_n_est(metrics["rows"])
    zip_path = zip_and_remove_predictions(prediction_dir, label)
    metrics["checkpoint"] = {
        "label": label,
        "candidate": candidate,
        "epoch": epoch,
        "weights": str(weights_path),
        "weights_sha256": sha256_file(weights_path),
    }
    metrics["aggregate_node_counts"] = {
        "n_pred": n_pred,
        "n_est": n_est,
        "n_pred_over_n_est": node_ratio,
    }
    metrics["prediction_zip"] = str(zip_path)
    metrics["wall_seconds"] = time.monotonic() - started
    atomic_write_text(metrics_path, json.dumps(metrics, ensure_ascii=False, indent=2))
    state = load_pipeline_state()
    evaluations = dict(state.get("evaluations", {}))
    evaluations[label] = {
        "status": "complete",
        "metrics": str(metrics_path),
        "score": metrics["summary"]["score"],
    }
    save_pipeline_state(status="evaluation_in_progress", active_evaluation=None, evaluations=evaluations)
    print(
        f"{label}: score={metrics['summary']['score']:.4f} edge={metrics['summary']['edge_jaccard']:.4f} "
        f"recall={metrics['summary']['node_recall']:.4f} N_pred/N_est={node_ratio:.4f}"
    )
    return metrics


if RUN_PIPELINE:
    completed_epochs, completed_payload = read_training_progress()
    if completed_epochs != EPOCHS:
        raise RuntimeError("Evaluation cannot start before all 20 training epochs are complete")
    del completed_payload
    internal_best_epoch = load_pipeline_state().get("internal_best_epoch")
    evaluation_specs = [
        *[
            {
                "label": f"epoch_{epoch:03d}",
                "weights": TRAIN_OUTPUT / f"edge_predictor_epoch_{epoch:03d}.pth",
                "candidate": True,
                "epoch": epoch,
            }
            for epoch in MILESTONE_EPOCHS
        ],
        {
            "label": "training_best",
            "weights": TRAIN_OUTPUT / "edge_predictor_best.pth",
            "candidate": True,
            "epoch": internal_best_epoch,
        },
    ]
    for reference_label, dataset in REFERENCE_SPECS.items():
        reference_metrics = load_valid_metrics(EVALUATION_ROOT / reference_label / "official_metrics.json")
        reference_weights = (
            REFERENCE_ROOT / reference_label / "edge_predictor_best.pth"
            if evaluation_is_complete(reference_metrics)
            else ensure_reference_model(reference_label, dataset)
        )
        evaluation_specs.append(
            {"label": reference_label, "weights": reference_weights, "candidate": False, "epoch": None}
        )

    all_metrics = {}
    for spec in evaluation_specs:
        all_metrics[spec["label"]] = evaluate_weights(
            spec["label"], spec["weights"], candidate=spec["candidate"], epoch=spec["epoch"]
        )
    save_pipeline_state(status="evaluation_complete", active_evaluation=None)
    print("All five candidate entries and both pilkwang references have official 12/12 LOEO metrics.")
else:
    print("BIOHUB_STEP1_RUN_PIPELINE=0: checkpoint evaluation intentionally skipped.")


In [ ]:
# Select by the official metric, package the artifact, upload weights, update reports, commit, and push.


def compact_metric(label: str, payload: dict) -> dict:
    summary = payload["summary"]
    nodes = payload["aggregate_node_counts"]
    checkpoint = payload["checkpoint"]
    return {
        "label": label,
        "candidate": bool(checkpoint["candidate"]),
        "epoch": checkpoint["epoch"],
        "weights_sha256": checkpoint["weights_sha256"],
        "score": float(summary["score"]),
        "adj_edge_jaccard": float(summary["adj_edge_jaccard"]),
        "edge_jaccard": float(summary["edge_jaccard"]),
        "division_jaccard": float(summary["division_jaccard"]),
        "division_tp": int(summary["division_tp"]),
        "division_fp": int(summary["division_fp"]),
        "division_fn": int(summary["division_fn"]),
        "node_recall": float(summary["node_recall"]),
        "n_pred": int(nodes["n_pred"]),
        "n_est": float(nodes["n_est"]),
        "n_pred_over_n_est": float(nodes["n_pred_over_n_est"]),
        "videos": int(summary["n"]),
        "wall_seconds": float(payload.get("wall_seconds", 0.0)),
    }


def upload_best_weights(weights_path: Path, config_path: Path, result_summary: dict) -> None:
    current_state = load_pipeline_state()
    if current_state.get("weights_uploaded"):
        print(f"Weights already uploaded as {WEIGHTS_DATASET_SUBFOLDER}/; skipping")
        return
    stage = RUN_DIR / "weights_dataset_stage"
    if stage.exists():
        if not stage.resolve().is_relative_to(RUN_DIR.resolve()):
            raise RuntimeError(f"Refusing to replace unexpected staging path: {stage}")
        shutil.rmtree(stage)
    stage.mkdir(parents=True)
    upload_env = {**os.environ, "PYTHONUTF8": "1", "PYTHONIOENCODING": "utf-8"}
    run_streamed(
        [
            sys.executable, "-m", "kaggle", "datasets", "download", WEIGHTS_DATASET,
            "-p", str(stage), "--unzip", "-o",
        ],
        cwd=PROJECT_ROOT,
        env=upload_env,
        log_path=RUN_DIR / "weights_upload.log",
        label="download_current_weights_dataset",
    )
    destination = stage / WEIGHTS_DATASET_SUBFOLDER
    destination.mkdir(parents=True, exist_ok=True)
    shutil.copy2(weights_path, destination / "edge_predictor_best.pth")
    shutil.copy2(config_path, destination / "config.json")
    if sha256_file(destination / "edge_predictor_best.pth") != sha256_file(weights_path):
        raise RuntimeError("Staged checkpoint hash mismatch")
    atomic_write_text(
        stage / "dataset-metadata.json",
        json.dumps({"id": WEIGHTS_DATASET, "title": "biohub-ft-weights"}, indent=2) + "\n",
    )
    readme = stage / "README.md"
    readme_text = readme.read_text(encoding="utf-8") if readme.exists() else "Biohub team weights.\n"
    marker = f"`{WEIGHTS_DATASET_SUBFOLDER}/`"
    if marker not in readme_text:
        readme_text = readme_text.rstrip() + (
            f"\n\n{marker} is Friend A STEP 1's official-best scratch checkpoint trained on 126 6bba videos "
            f"and selected on 12 unseen 44b6 videos: score {result_summary['score']:.6f}, "
            f"edge Jaccard {result_summary['edge_jaccard']:.6f}, node recall {result_summary['node_recall']:.6f}, "
            f"N_pred/N_est {result_summary['n_pred_over_n_est']:.6f}. No leaderboard submission was made.\n"
        )
        atomic_write_text(readme, readme_text)
    run_streamed(
        [
            sys.executable, "-m", "kaggle", "datasets", "version", "-p", str(stage),
            "--dir-mode", "zip", "-m", f"{EXPERIMENT}: official-best LOEO checkpoint",
        ],
        cwd=PROJECT_ROOT,
        env=upload_env,
        log_path=RUN_DIR / "weights_upload.log",
        label="upload_best_weights",
    )
    required_remote_files = [
        f"{WEIGHTS_DATASET_SUBFOLDER}/edge_predictor_best.pth",
        f"{WEIGHTS_DATASET_SUBFOLDER}/config.json",
    ]
    listing = ""
    for attempt in range(6):
        listing = run_checked(
            [sys.executable, "-m", "kaggle", "datasets", "files", WEIGHTS_DATASET, "--page-size", "200"],
            cwd=PROJECT_ROOT,
            env=upload_env,
        )
        if all(name in listing for name in required_remote_files):
            break
        if attempt < 5:
            print("Kaggle version is still processing; checking again in 10 seconds...")
            time.sleep(10)
    else:
        raise RuntimeError("Kaggle accepted the version, but the expected STEP-1 files are not visible after 60 seconds")
    save_pipeline_state(
        status="weights_uploaded",
        weights_uploaded=True,
        weights_dataset=WEIGHTS_DATASET,
        weights_dataset_subfolder=WEIGHTS_DATASET_SUBFOLDER,
        uploaded_checkpoint_sha256=sha256_file(weights_path),
    )
    print(f"Uploaded and verified: {WEIGHTS_DATASET}/{WEIGHTS_DATASET_SUBFOLDER}")


def update_results_markdown(result: dict) -> None:
    path = PROJECT_ROOT / "experiments" / "RESULTS.md"
    text = path.read_text(encoding="utf-8")
    best = result["official_best"]
    refs = result["references"]
    train_hours = result["timing"]["training_gpu_seconds"] / 3600
    eval_hours = result["timing"]["evaluation_wall_seconds"] / 3600
    line = (
        f"| local Friend A exp02 | scratch 20ep, 126x6bba -> LOEO-44b6; BCE, w2, ds1,4,4, cosine 1e-4->1e-5, seed42 | - | "
        f"best **{best['score']:.4f}** at {best['label']} (edge {best['edge_jaccard']:.4f}, recall {best['node_recall']:.4f}, "
        f"Npred/Nest {best['n_pred_over_n_est']:.3f}); primary ref {refs['reference_pilkwang_primary']['score']:.4f}, "
        f"secondary ref {refs['reference_pilkwang_secondary_seed314159']['score']:.4f} | "
        f"{train_hours:.1f} h train + {eval_hours:.1f} h eval | 12/12 honest videos; milestones 5/10/15/20 + training-best; "
        f"dataset `{WEIGHTS_DATASET_SUBFOLDER}/`; stop delta20-15={result['decision']['score20_minus_score15']:+.4f}; no LB submission |"
    )
    lines = [existing for existing in text.splitlines() if not existing.startswith("| local Friend A exp02 |")]
    insert_at = next((index for index, existing in enumerate(lines) if existing.startswith("## Per-video")), len(lines))
    lines.insert(insert_at, line)
    atomic_write_text(path, "\n".join(lines).rstrip() + "\n")


def update_context(result: dict) -> None:
    path = PROJECT_ROOT / "CONTEXT.md"
    text = path.read_text(encoding="utf-8")
    marker = "<!-- friend-a-exp02-loeo20-result -->"
    best = result["official_best"]
    decision = result["decision"]
    note = (
        f"{marker}\n- {datetime.now().date().isoformat()} Friend A STEP 1 complete: 20-epoch scratch 6bba->44b6 official best "
        f"{best['score']:.4f} ({best['label']}, edge {best['edge_jaccard']:.4f}, recall {best['node_recall']:.4f}, "
        f"N_pred/N_est {best['n_pred_over_n_est']:.3f}); delta20-15 {decision['score20_minus_score15']:+.4f}; "
        f"decision `{decision['next_step']}`; weights `{WEIGHTS_DATASET_SUBFOLDER}/`, no LB submission."
    )
    pattern = re.compile(re.escape(marker) + r"\n- [^\n]*")
    if pattern.search(text):
        text = pattern.sub(note, text, count=1)
    else:
        heading = next(
            (line for line in text.splitlines() if line.startswith("## 14. Operational state")),
            None,
        )
        if heading is None:
            raise RuntimeError("CONTEXT.md §14 heading not found")
        text = text.replace(heading, heading + "\n" + note, 1)
    atomic_write_text(path, text.rstrip() + "\n")


def commit_and_push_reports() -> None:
    expected_paths = {
        "CONTEXT.md",
        "experiments/RESULTS.md",
        f"experiments/{EXPERIMENT}/results.json",
    }
    status_text = run_checked([*GIT, "status", "--porcelain"], cwd=PROJECT_ROOT)
    changed_paths = set()
    for line in status_text.splitlines():
        path_text = line[3:].strip()
        if " -> " in path_text:
            path_text = path_text.split(" -> ", 1)[1]
        changed_paths.add(path_text.replace("\\", "/"))
    unexpected = changed_paths - expected_paths
    if unexpected:
        raise RuntimeError("Refusing auto-commit because unrelated paths are modified: " + ", ".join(sorted(unexpected)))
    run_checked([*GIT, "add", "--", *sorted(expected_paths)], cwd=PROJECT_ROOT)
    staged = subprocess.run([*GIT, "diff", "--cached", "--quiet"], cwd=PROJECT_ROOT)
    if staged.returncode == 1:
        message = f"Record Friend A STEP 1 honest LOEO result"
        output = run_checked([*GIT, "commit", "-m", message], cwd=PROJECT_ROOT)
        if output.strip():
            print(output.strip())
    elif staged.returncode != 0:
        raise RuntimeError("git diff --cached --quiet failed")
    output = run_checked([*GIT, "push", "origin", EXPECTED_BRANCH], cwd=PROJECT_ROOT)
    if output.strip():
        print(output.strip())
    save_pipeline_state(status="complete", git_pushed=True)


if RUN_PIPELINE:
    metric_payloads = {}
    for path in sorted(EVALUATION_ROOT.glob("*/official_metrics.json")):
        payload = load_valid_metrics(path)
        if payload is not None and "checkpoint" in payload and "aggregate_node_counts" in payload:
            metric_payloads[path.parent.name] = payload
    required_labels = {
        *[f"epoch_{epoch:03d}" for epoch in MILESTONE_EPOCHS],
        "training_best",
        *REFERENCE_SPECS.keys(),
    }
    missing_labels = required_labels - set(metric_payloads)
    if missing_labels:
        raise RuntimeError("Cannot publish; evaluations are missing: " + ", ".join(sorted(missing_labels)))
    compact = {label: compact_metric(label, payload) for label, payload in metric_payloads.items()}
    candidate_labels = [f"epoch_{epoch:03d}" for epoch in MILESTONE_EPOCHS] + ["training_best"]
    official_best_label = max(
        candidate_labels,
        key=lambda label: (compact[label]["score"], -(compact[label]["epoch"] or 10**9)),
    )
    official_best = compact[official_best_label]
    selected_source = Path(metric_payloads[official_best_label]["checkpoint"]["weights"])
    config_source = selected_source.with_name("config.json")
    ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)
    selected_target = ARTIFACT_DIR / "edge_predictor_best.pth"
    config_target = ARTIFACT_DIR / "config.json"
    shutil.copy2(selected_source, selected_target)
    shutil.copy2(config_source, config_target)
    shutil.copy2(RUN_CONFIG_PATH, ARTIFACT_DIR / "run_config.json")
    shutil.copy2(RUN_DIR / "training_history.csv", ARTIFACT_DIR / "training_history.csv")
    shutil.copy2(RUN_DIR / "training_history.json", ARTIFACT_DIR / "training_history.json")
    if sha256_file(selected_source) != sha256_file(selected_target):
        raise RuntimeError("Packaged official-best checkpoint hash mismatch")

    score15 = compact["epoch_015"]["score"]
    score20 = compact["epoch_020"]["score"]
    delta20_15 = score20 - score15
    if official_best["score"] >= 0.85:
        next_step = "STEP_3_validator_usable"
    elif official_best["score"] < 0.80:
        next_step = "STEP_4_dense_labels_sparse_limit"
    else:
        next_step = "STEP_2_borderline_review_0.80_to_0.85"
    training_history = json.loads((RUN_DIR / "training_history.json").read_text(encoding="utf-8"))
    result = {
        "experiment": EXPERIMENT,
        "step": 1,
        "status": "complete_official_loeo",
        "completed_at": datetime.now().astimezone().isoformat(),
        "split": "126x6bba train -> 12x44b6 validation",
        "initialization": "scratch",
        "recipe": json.loads(RUN_CONFIG_PATH.read_text(encoding="utf-8")),
        "checkpoint_table": [compact[label] for label in candidate_labels],
        "references": {label: compact[label] for label in REFERENCE_SPECS},
        "official_best": official_best,
        "official_best_checkpoint_sha256": sha256_file(selected_target),
        "artifact_dir": str(ARTIFACT_DIR),
        "weights_dataset": WEIGHTS_DATASET,
        "weights_dataset_subfolder": WEIGHTS_DATASET_SUBFOLDER,
        "decision": {
            "score20_minus_score15": delta20_15,
            "twenty_epochs_enough_by_stop_rule": delta20_15 < 0.005,
            "best_score_gate": official_best["score"],
            "next_step": next_step,
        },
        "timing": {
            "training_gpu_seconds": sum(row["train_seconds"] + row["validation_seconds"] for row in training_history),
            "evaluation_wall_seconds": sum(row["wall_seconds"] for row in compact.values()),
        },
        "official_metric": "adjusted_edge_jaccard + 0.1 * division_jaccard",
        "leaderboard_submission": False,
    }
    atomic_write_text(RUN_DIR / "results.json", json.dumps(result, ensure_ascii=False, indent=2))
    atomic_write_text(ARTIFACT_DIR / "results.json", json.dumps(result, ensure_ascii=False, indent=2))

    if AUTO_UPLOAD:
        upload_best_weights(selected_target, config_target, official_best)
    else:
        print("BIOHUB_STEP1_AUTO_UPLOAD=0: Kaggle upload intentionally skipped")
    result["weights_uploaded"] = bool(load_pipeline_state().get("weights_uploaded"))
    atomic_write_text(RUN_DIR / "results.json", json.dumps(result, ensure_ascii=False, indent=2))
    atomic_write_text(ARTIFACT_DIR / "results.json", json.dumps(result, ensure_ascii=False, indent=2))
    tracked_results = EXPERIMENT_DIR / "results.json"
    atomic_write_text(tracked_results, json.dumps(result, ensure_ascii=False, indent=2))
    update_results_markdown(result)
    update_context(result)
    save_pipeline_state(status="reports_written", official_best_label=official_best_label)

    if AUTO_GIT:
        if AUTO_UPLOAD and not result["weights_uploaded"]:
            raise RuntimeError("Refusing to commit the final report before the requested weights upload succeeds")
        commit_and_push_reports()
    else:
        print("BIOHUB_STEP1_AUTO_GIT=0: report files were written but not committed")
    print(
        f"STEP 1 COMPLETE: best={official_best_label} score={official_best['score']:.6f}; "
        f"next={next_step}; no leaderboard submission."
    )
else:
    print("BIOHUB_STEP1_RUN_PIPELINE=0: packaging/publication intentionally skipped.")
